# CNN-ResBiGRU-SE for sEMG-based Hand Gesture Recognition
## NinaPro-DB1 Dataset

This notebook provides the complete implementation of the **CNN-ResBiGRU-SE** framework
for surface electromyography (sEMG)-based hand gesture recognition (HGR) using the
publicly available **NinaPro-DB1** benchmark dataset.

---

### Architecture Overview
```
sEMG Input (window × channels)
        ↓
  [ConvB Block 1]  ← Conv1D-BN-ReLU × 3 (strided)
        ↓
  [ConvB Block 2]  ← Conv1D-BN-ReLU × 3 (strided)
        ↓
  [ResBiGRU Block] ← BiGRU + LayerNorm + Residual
        ↓
  [SE Block]       ← Squeeze-and-Excitation (channel attention)
        ↓
  [GAP + Dense]    ← Global Average Pooling + SoftMax
        ↓
  Gesture Label
```

### Dataset: NinaPro-DB1
| Property | Value |
|---|---|
| Subjects | 27 non-disabled |
| Channels | 10 (OttoBock MyoBock 13E200-50) |
| Gestures | 52 (Exercise A: 12, B: 17, C: 23) + rest |
| Repetitions | 10 per gesture |
| Sampling Rate | 100 Hz |

### Reference
> S. Mekruksavanich, N. Hnoohom, and A. Jitpattanakul, *"Deep Residual Network with
> Channel Attention for Improving Hand Gesture Recognition with Surface Electromyography
> Signal,"* IEEE Access, 2024.


In [ ]:
# ─── Cell 1: Install dependencies ────────────────────────────────────────────
!pip install scipy scikit-learn seaborn tensorflow --quiet
print('Dependencies ready.')


In [ ]:
# ─── Cell 2: Imports ─────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')


In [ ]:
# ─── Cell 3: Configuration ────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════
#  Modify the parameters below to replicate specific experiment configurations
# ═══════════════════════════════════════════════════════════════════════════════

# ── Dataset settings ──
NUM_SUBJECTS   = 27
NUM_CHANNELS   = 10        # OttoBock electrodes
SAMPLING_RATE  = 100       # Hz
EXERCISE_SIZES = {'E1': 12, 'E2': 17, 'E3': 23}

# ── Experiment: select exercise subset ──
# Options: 'E1' | 'E2' | 'E3' | 'E1+E2+E3'
EXERCISE = 'E1+E2+E3'      # Full 52-gesture set (as reported in the manuscript)

# ── Windowing (200 ms window / 50 ms stride at 100 Hz) ──
WINDOW_SIZE  = 20          # samples  (200 ms × 100 Hz)
WINDOW_STRIDE = 5          # samples  ( 50 ms × 100 Hz)

# ── Model hyperparameters ──
CONV_FILTERS_1  = 64
CONV_FILTERS_2  = 128
GRU_UNITS       = 128      # BiGRU output = 2 × GRU_UNITS = 256
SE_REDUCTION    = 16
L2_LAMBDA       = 1e-4

# ── Training hyperparameters (Table 2 in manuscript) ──
N_FOLDS        = 5
N_EPOCHS       = 200
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
BETA_1, BETA_2 = 0.9, 0.999
EPSILON        = 1e-8
PATIENCE       = 20        # Early-stopping patience

# ── Output directory ──
OUTPUT_DIR = '/content/results_db1'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration loaded.')


In [ ]:
# ─── Cell 4: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# !! UPDATE THIS PATH to where you placed the NinaPro-DB1 folder on your Drive !!
# Expected structure:
#   DB1_PATH/
#     S1/  S1_A1_E1.mat  S1_A1_E2.mat  S1_A1_E3.mat
#     S2/  S2_A1_E1.mat  ...
#     ...
#     S27/ S27_A1_E1.mat ...
DB1_PATH = '/content/drive/MyDrive/NinaPro/DB1'   # <── update this

if not os.path.isdir(DB1_PATH):
    raise FileNotFoundError(
        f'Dataset not found at: {DB1_PATH}\n'
        'Download NinaPro-DB1 from https://ninapro.hevs.ch/instructions/DB1.html\n'
        'and update DB1_PATH above.'
    )
print(f'Dataset path OK: {DB1_PATH}')


In [ ]:
# ─── Cell 5: Data Loading ─────────────────────────────────────────────────────
def load_subject_db1(data_path, subject_id, exercises):
    """
    Load NinaPro-DB1 .mat files for one subject and concatenate
    the requested exercises.

    Args:
        data_path  : root directory of DB1
        subject_id : integer 1..27
        exercises  : list such as ['E1','E2','E3']

    Returns:
        emg         : (N, 10)  raw sEMG amplitude, float32
        labels      : (N,)     gesture labels (1-indexed; 0 = rest)
        repetitions : (N,)     repetition numbers (1-indexed)
    """
    emg_all, lbl_all, rep_all = [], [], []
    label_offset = 0

    for ex in exercises:
        ex_num   = {'E1': 1, 'E2': 2, 'E3': 3}[ex]
        filename = f'S{subject_id}_A1_E{ex_num}.mat'

        # Try subject sub-folder first, then flat directory
        for candidate in [
            os.path.join(data_path, f'S{subject_id}', filename),
            os.path.join(data_path, filename)
        ]:
            if os.path.isfile(candidate):
                filepath = candidate
                break
        else:
            raise FileNotFoundError(f'Cannot find {filename} in {data_path}')

        mat     = sio.loadmat(filepath)
        emg     = mat['emg'].astype(np.float32)          # (N, 10)
        labels  = mat['restimulus'].flatten().astype(int) # (N,)
        reps    = mat['rerepetition'].flatten().astype(int)# (N,)

        # Offset non-rest gesture labels so they are unique across exercises
        non_rest = labels > 0
        labels[non_rest] += label_offset
        label_offset += EXERCISE_SIZES[ex]

        emg_all.append(emg)
        lbl_all.append(labels)
        rep_all.append(reps)

    return (
        np.concatenate(emg_all,  axis=0),
        np.concatenate(lbl_all,  axis=0),
        np.concatenate(rep_all,  axis=0)
    )

print('load_subject_db1() defined.')


In [ ]:
# ─── Cell 6: Preprocessing ───────────────────────────────────────────────────
def preprocess_emg_db1(emg):
    """
    Preprocessing for NinaPro-DB1.

    The OttoBock MyoBock 13E200-50 sensors apply internal hardware
    conditioning (band-pass amplification, full-wave rectification,
    and RMS smoothing) before outputting at 100 Hz.  Therefore, no
    additional software band-pass filter is applied here.

    Pipeline:
        1. Zero-mean, unit-variance normalisation (per channel, per subject)

    Args:
        emg : (N, 10) float32

    Returns:
        emg_norm : (N, 10) float32, normalised
    """
    mu  = emg.mean(axis=0, keepdims=True)
    std = emg.std(axis=0,  keepdims=True) + 1e-8
    return ((emg - mu) / std).astype(np.float32)


def segment_windows(emg, labels, repetitions, window_size, stride):
    """
    Segment a continuous sEMG recording into fixed-length windows.

    Only windows that contain a single, unique, non-rest gesture label
    are kept (boundary windows are discarded).

    Args:
        emg         : (N, C) preprocessed sEMG
        labels      : (N,)   gesture labels (0 = rest)
        repetitions : (N,)   repetition numbers
        window_size : int    number of time samples per window
        stride      : int    shift in samples between successive windows

    Returns:
        X    : (M, window_size, C)  gesture windows
        y    : (M,)                 0-indexed gesture labels
        reps : (M,)                 repetition number of each window
    """
    X_list, y_list, r_list = [], [], []

    for start in range(0, len(emg) - window_size + 1, stride):
        end    = start + window_size
        seg_lbl = labels[start:end]
        active  = seg_lbl[seg_lbl > 0]          # exclude rest frames

        # Keep only pure-gesture windows (all frames same label)
        unique_active = np.unique(active)
        if len(unique_active) == 1:
            X_list.append(emg[start:end])
            y_list.append(int(unique_active[0]) - 1)   # convert to 0-index
            r_list.append(int(repetitions[start + window_size // 2]))

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.int32),
        np.array(r_list, dtype=np.int32)
    )

print('Preprocessing functions defined.')


In [ ]:
# ─── Cell 7: CNN-ResBiGRU-SE Architecture ────────────────────────────────────
# =============================================================================
#  Building blocks
# =============================================================================

def conv_block(x, num_filters, kernel_size=3, use_stride=True, l2=1e-4):
    """
    Convolutional Block  (ConvB).
    Structure: [Conv1D → BN → ReLU] × 3
    The third Conv1D uses stride=2 to halve temporal resolution.
    """
    stride = 2 if use_stride else 1
    for i, s in enumerate([1, 1, stride]):
        x = layers.Conv1D(
                num_filters, kernel_size,
                strides=s, padding='same',
                kernel_regularizer=regularizers.l2(l2),
                use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x


def resbigru_block(x, gru_units, l2=1e-4):
    """
    Residual Bidirectional GRU Block  (ResBiGRU).

    Equations (from manuscript, Eq. 3):
      x_t^{f(i+1)} = LN( x_t^{f(i)} + GRU_f(x_t^{f(i)}, h_{t-1}^f ; W^f) )
      x_t^{b(i+1)} = LN( x_t^{b(i)} + GRU_b(x_t^{b(i)}, h_{t+1}^b ; W^b) )
      y_t          = concat(x_t^f, x_t^b)
    """
    # Bidirectional GRU  →  output shape: (batch, T, 2*gru_units)
    bigru_out = layers.Bidirectional(
        layers.GRU(gru_units, return_sequences=True,
                   kernel_regularizer=regularizers.l2(l2)),
        name='bigru'
    )(x)

    # Projection for residual (only if dimensions mismatch)
    if x.shape[-1] != bigru_out.shape[-1]:
        residual = layers.Dense(bigru_out.shape[-1], use_bias=False,
                                name='res_proj')(x)
    else:
        residual = x

    # Layer Normalisation + residual connection (Eq. 2 corrected: with epsilon)
    # LN: x_hat = (x - mu) / sqrt(var + eps) * gamma + beta
    out = layers.LayerNormalization(epsilon=1e-8, name='resbigru_ln')(residual + bigru_out)
    return out


def se_block(x, reduction_ratio=16):
    """
    Squeeze-and-Excitation Block  (SE).

    Equations (from manuscript, Eq. 4 & 5):
      Squeeze  : Z_c = (1/T) * sum_t U_c(t)          [GAP over time]
      Excitation: s   = sigmoid( W2 * ReLU( W1 * z ) )
      Output   : U_hat_c = s_c * U_c                 [channel reweighting]
    """
    C = x.shape[-1]
    r = max(1, C // reduction_ratio)

    z = layers.GlobalAveragePooling1D(name='se_gap')(x)             # (batch, C)
    s = layers.Dense(r,  activation='relu',    name='se_fc1')(z)
    s = layers.Dense(C,  activation='sigmoid', name='se_fc2')(s)
    s = layers.Reshape((1, C), name='se_reshape')(s)
    return layers.Multiply(name='se_scale')([x, s])                 # (batch, T, C)


# =============================================================================
#  Full model
# =============================================================================

def build_cnn_resbigru_se(input_shape, num_classes,
                           conv_f1=64, conv_f2=128,
                           gru_units=128, se_r=16, l2=1e-4):
    """
    Build CNN-ResBiGRU-SE.

    Data flow:
        Input(T, C)
          → ConvB-1(T/2,  64)
          → ConvB-2(T/4, 128)
          → ResBiGRU(T/4, 256)
          → SE(T/4, 256)
          → GAP(256)
          → Dense(num_classes, softmax)

    Args:
        input_shape : (window_size, num_channels)  e.g. (20, 10) for DB1
        num_classes : number of gesture categories  e.g. 52
    """
    inp = keras.Input(shape=input_shape, name='sEMG_input')

    # ── Spatial feature extraction ──────────────────────────────────
    x = conv_block(inp, conv_f1, kernel_size=3, use_stride=True, l2=l2)
    x = conv_block(x,   conv_f2, kernel_size=3, use_stride=True, l2=l2)

    # ── Temporal feature extraction ──────────────────────────────────
    x = resbigru_block(x, gru_units, l2=l2)

    # ── Channel attention ────────────────────────────────────────────
    x = se_block(x, reduction_ratio=se_r)

    # ── Classification head ──────────────────────────────────────────
    x   = layers.GlobalAveragePooling1D(name='gap')(x)
    out = layers.Dense(num_classes, activation='softmax',
                       name='gesture_output')(x)

    return Model(inputs=inp, outputs=out, name='CNN_ResBiGRU_SE')


# Quick sanity check
demo = build_cnn_resbigru_se(
    input_shape=(WINDOW_SIZE, NUM_CHANNELS), num_classes=52)
demo.summary()


In [ ]:
# ─── Cell 8: Training Function (5-fold CV at repetition level) ───────────────
def train_subject_kfold(X, y, reps, num_classes,
                         n_folds=5, n_epochs=200, batch_size=32):
    """
    Train CNN-ResBiGRU-SE for one subject using k-fold CV.

    Cross-validation strategy (prevents data leakage):
        - Folds are defined at the *repetition* level.
        - Window segmentation is applied independently within each
          train/test partition; no window from a test repetition
          appears in training.

    Args:
        X           : (M, window_size, channels)  gesture windows
        y           : (M,)  0-indexed gesture labels
        reps        : (M,)  repetition number per window
        num_classes : int

    Returns:
        mean_acc  : float  mean accuracy across folds (%)
        mean_f1   : float  mean macro F1 across folds (%)
        all_true  : list   ground-truth labels (all folds)
        all_pred  : list   predicted labels    (all folds)
    """
    unique_reps = np.unique(reps)
    kf          = KFold(n_splits=n_folds, shuffle=False)

    fold_accs, fold_f1s = [], []
    all_true, all_pred  = [], []

    for fold, (tr_idx, te_idx) in enumerate(kf.split(unique_reps)):
        train_reps = unique_reps[tr_idx]
        test_reps  = unique_reps[te_idx]

        tr_mask = np.isin(reps, train_reps)
        te_mask = np.isin(reps, test_reps)

        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_te, y_te = X[te_mask], y[te_mask]

        y_tr_cat = tf.keras.utils.to_categorical(y_tr, num_classes)
        y_te_cat = tf.keras.utils.to_categorical(y_te, num_classes)

        # Build fresh model for each fold
        model = build_cnn_resbigru_se(
            input_shape=X_tr.shape[1:],
            num_classes=num_classes,
            conv_f1=CONV_FILTERS_1, conv_f2=CONV_FILTERS_2,
            gru_units=GRU_UNITS, se_r=SE_REDUCTION, l2=L2_LAMBDA
        )

        # Adam optimiser (Table 2 in manuscript)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=LEARNING_RATE,
                beta_1=BETA_1, beta_2=BETA_2, epsilon=EPSILON
            ),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=PATIENCE,
                restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=10,
                min_lr=1e-6, verbose=0)
        ]

        model.fit(
            X_tr, y_tr_cat,
            validation_data=(X_te, y_te_cat),
            epochs=n_epochs, batch_size=batch_size,
            callbacks=callbacks, verbose=0
        )

        y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
        acc    = accuracy_score(y_te, y_pred) * 100
        f1     = f1_score(y_te, y_pred, average='macro', zero_division=0) * 100

        fold_accs.append(acc)
        fold_f1s.append(f1)
        all_true.extend(y_te.tolist())
        all_pred.extend(y_pred.tolist())

        print(f'    Fold {fold+1}/{n_folds}  Acc={acc:.2f}%  F1={f1:.2f}%')

    return np.mean(fold_accs), np.mean(fold_f1s), all_true, all_pred

print('Training function defined.')


In [ ]:
# ─── Cell 9: Main Training Loop ───────────────────────────────────────────────

# Determine exercise list and number of classes
if EXERCISE == 'E1+E2+E3':
    ex_list, num_classes = ['E1', 'E2', 'E3'], 52
elif EXERCISE == 'E1':
    ex_list, num_classes = ['E1'], 12
elif EXERCISE == 'E2':
    ex_list, num_classes = ['E2'], 17
elif EXERCISE == 'E3':
    ex_list, num_classes = ['E3'], 23
else:
    raise ValueError(f'Unknown EXERCISE: {EXERCISE}')

print(f'Dataset      : NinaPro-DB1')
print(f'Exercise     : {EXERCISE}  ({num_classes} classes)')
print(f'Window/Stride: {WINDOW_SIZE}/{WINDOW_STRIDE} samples '
      f'({WINDOW_SIZE/SAMPLING_RATE*1000:.0f}/{WINDOW_STRIDE/SAMPLING_RATE*1000:.0f} ms)')
print('='*60)

subject_accs, subject_f1s = [], []
all_results = []

for subj in range(1, NUM_SUBJECTS + 1):
    print(f'\nSubject {subj:02d}/{NUM_SUBJECTS}')

    # 1. Load raw data
    emg, labels, repetitions = load_subject_db1(DB1_PATH, subj, ex_list)

    # 2. Normalise
    emg_norm = preprocess_emg_db1(emg)

    # 3. Segment into windows
    X, y, reps = segment_windows(
        emg_norm, labels, repetitions, WINDOW_SIZE, WINDOW_STRIDE)

    if len(X) == 0:
        print(f'  [WARNING] No valid windows for subject {subj} — skipped.')
        continue

    print(f'  Windows  : {X.shape}   Classes: {np.unique(y).size}')

    # 4. Train + evaluate with 5-fold repetition-level CV
    acc, f1, y_true, y_pred = train_subject_kfold(
        X, y, reps, num_classes,
        n_folds=N_FOLDS, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE
    )

    subject_accs.append(acc)
    subject_f1s.append(f1)
    all_results.append({'Subject': subj, 'Accuracy': acc, 'F1_Score': f1})
    print(f'  ► Subject {subj:02d}  Acc={acc:.2f}%  F1={f1:.2f}%')

# ── Summary ──
mean_acc = np.mean(subject_accs)
mean_f1  = np.mean(subject_f1s)
print('\n' + '='*60)
print(f'OVERALL ({EXERCISE})')
print(f'  Avg Accuracy : {mean_acc:.2f}% ± {np.std(subject_accs):.2f}%')
print(f'  Avg F1-Score : {mean_f1:.2f}% ± {np.std(subject_f1s):.2f}%')
print('='*60)

# Save CSV
results_df = pd.DataFrame(all_results)
csv_path = os.path.join(OUTPUT_DIR, f'results_{EXERCISE}.csv')
results_df.to_csv(csv_path, index=False)
print(f'Results saved → {csv_path}')


In [ ]:
# ─── Cell 10: Results Visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'CNN-ResBiGRU-SE  |  NinaPro-DB1  |  {EXERCISE}', fontsize=13, fontweight='bold')

# ── Box plot ──
bp = axes[0].boxplot(
    subject_accs, patch_artist=True, showmeans=True,
    boxprops=dict(facecolor='#AED6F1'),
    medianprops=dict(color='navy', linewidth=2),
    meanprops=dict(marker='x', markerfacecolor='red', markeredgecolor='red', markersize=10)
)
axes[0].axhline(mean_acc, color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {mean_acc:.2f}%')
axes[0].set_title('Accuracy Distribution (all subjects)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_xlabel('All Subjects Combined')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Per-subject bar chart ──
colors = ['#2E86C1' if a >= mean_acc else '#E74C3C' for a in subject_accs]
axes[1].bar(range(1, len(subject_accs)+1), subject_accs, color=colors, edgecolor='k', linewidth=0.5)
axes[1].axhline(mean_acc, color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {mean_acc:.2f}%')
axes[1].set_title('Per-Subject Recognition Accuracy')
axes[1].set_xlabel('Subject ID')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, f'accuracy_{EXERCISE}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')
